# Train Cow Ear Tag Detector YOLO26n 🐮

This notebook fine-tunes Ultralytics YOLO26n for cow ear tag detection using the [CEID-D dataset](https://www.kaggle.com/datasets/fandaoerji/cow-eartag-detection-dataset/data), which was pre-processed prior to training.

**Notebook Overview:**
- Loads the pre-processed YOLO-format dataset
- Fine-tunes YOLO26n using the [Ultralytics framework](https://github.com/ultralytics/ultralytics).
- Logs training metrics and artifacts
- Saves the trained model weights

**Base Model:** [Ultralytics YOLO26n](https://arxiv.org/abs/2606.03748) \
**Trained Model:** [Cow Ear Tag Detector YOLO26n 🐮](https://huggingface.co/mirandamurphy/cow-ear-tag-detector-yolo26n)

**Dataset Source:** CEID-D (Cow Ear Tag Detection Dataset), available on [Kaggle](https://www.kaggle.com/datasets/fandaoerji/cow-eartag-detection-dataset/data).

The original CEID-D dataset contained 2,675 images and 2,451 YOLO label files. After preprocessing, 2,196 images and labels remained.

The dataset was split **70/15/15** into train, validation, and test sets.

| Split | # of images |
|-------|-------------|
| Train | 1537        |
| Val   | 329         |
| Test  | 330         |

**Author:** Miranda Murphy \
**Contact:** mirandamurphy.dev@protonmail.com \
**License:** AGPL-3.0

For full dataset pre-processing details, model information, and licensing, see the repository README and [model card](https://huggingface.co/mirandamurphy/cow-ear-tag-detector-yolo26n).



In [ ]:
%pip install -q dagshub mlflow ultralytics

import os
from pathlib import Path

import dagshub
import mlflow
from ultralytics import settings
from ultralytics import YOLO
from google.colab import userdata

In [ ]:
# Load repo credentials from Colab Secrets
DAGSHUB_USER = userdata.get("DAGSHUB_USER")
DAGSHUB_REPO = userdata.get("DAGSHUB_REPO")
DAGSHUB_TOKEN = userdata.get("DAGSHUB_TOKEN")

REMOTE_PATH = userdata.get("REMOTE_PATH")
LOCAL_PATH = userdata.get("LOCAL_PATH")

os.environ["DAGSHUB_USER"] = DAGSHUB_USER
os.environ["DAGSHUB_REPO"] = DAGSHUB_REPO
os.environ["DAGSHUB_TOKEN"] = DAGSHUB_TOKEN

os.environ["REMOTE_PATH"] = REMOTE_PATH
os.environ["LOCAL_PATH"] = LOCAL_PATH

In [ ]:
# Download preprocessed dataset into the Colab VM
!dagshub download --bucket "$DAGSHUB_USER/$DAGSHUB_REPO" "$REMOTE_PATH" "$LOCAL_PATH"

In [ ]:
DATASET_DIR = Path(LOCAL_PATH)
DATA_YAML = DATASET_DIR / "data.yaml"

print(f"Dataset path: {DATASET_DIR}")
print(f"data.yaml exists: {DATA_YAML.exists()}")

In [ ]:
# Verify all the data was downloaded
for split in ["train", "val", "test"]:
    image_dir = DATASET_DIR / "images" / split
    if image_dir.exists():
        count = len(list(image_dir.glob("*")))
        print(f"Images in {split} split: {count}")
    else:
        print(f"Images in {split} split not found")

In [ ]:
# Setup MLflow tracking via DagsHub
EXPERIMENT_NAME = userdata.get("EXPERIMENT_NAME")
RUN_NAME = userdata.get("RUN_NAME")

os.environ["MLFLOW_EXPERIMENT_NAME"] = EXPERIMENT_NAME
os.environ["MLFLOW_RUN"] = RUN_NAME
os.environ["MLFLOW_KEEP_RUN_ACTIVE"] = "True"

settings.update({"mlflow": True})

dagshub.init(repo_owner=DAGSHUB_USER,
             repo_name=DAGSHUB_REPO,
             mlflow=True
             )

mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow tracking URI: ", mlflow.get_tracking_uri())

In [ ]:
# Train YOLO26n Model
model = YOLO("yolo26n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    warmup_epochs=0,
    patience=25,
    freeze=None,
    batch=-1,
    imgsz=640,
    save=True,
    save_period=5,
    cache="disk"
)

In [ ]:
# Validate trained model and log metrics
best_model_path = "/content/runs/detect/train/weights/best.pt"
best_model = YOLO(best_model_path)

metrics = best_model.val(data=str(DATA_YAML))

val_metrics = {
    "val_mean_precision": metrics.box.mp,
    "val_mean_recall": metrics.box.mr,
    "val_fitness": metrics.box.fitness(),
    "val_map50-95": metrics.box.map,
    "val_map50": metrics.box.map50,
    "val_map75": metrics.box.map75,
    "val_ms_image_preprocess": metrics.speed['preprocess'],
    "val_ms_image_inference": metrics.speed['inference'],
    "val_ms_image_loss": metrics.speed['loss'],
    "val_ms_image_postprocess": metrics.speed['postprocess']
}

mlflow.log_metrics(val_metrics)
mlflow.log_artifact(best_model_path, artifact_path="model_weights")

In [ ]:
run = mlflow.active_run()

if run:
    mlflow.end_run()
else:
    print("No active MLflow run found.")